In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) return false;

In [ ]:
import dataset, plots
import pandas as pd

sp500_components = dataset.SP500.load_historical()

### Dane cenowe z EODHD

Na EODHD dostępnych jest kilka tickerów, dla których można korzystać z danych bezpłatnie. W przypadku akcji amerykańskich są to:
- AAPL.US
- TSLA.US
- VTI.US
- AMZN.US

In [ ]:
api_key = "demo"
free_tickers = ["AAPL.US", "TSLA.US", "VTI.US", "AMZN.US"]

eodhd_data_free_tickers: dict[str, pd.DataFrame] = dataset.EODHD.download(
    tickers=free_tickers,
    api_key=api_key,
    save_csv=True
)

for column_name, frame in eodhd_data_free_tickers.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.name or 'Date'}: {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
tickers = dataset.tickers_from_sp500_components(sp500_components)

eodhd_data: dict[str, pd.DataFrame] = dataset.EODHD.download(
    tickers,
    save_csv=True,
    redownload_missing_tickers=True,
)

In [ ]:
for column_name, frame in eodhd_data.items():
    if frame.empty:
        print(f"{column_name}: empty")
    else:
        print(f"{column_name}: {frame.shape}, {frame.index.min().date()} -> {frame.index.max().date()}")

In [ ]:
coverage_df = plots.coverage_over_time(eodhd_data, sp500_components)
plots.summarize_df(coverage_df)

In [ ]:
from pathlib import Path

output_dir = dataset.EODHD.dst_dir
output_dir.mkdir(parents=True, exist_ok=True)

for column_name, frame in eodhd_data.items():
    frame.to_csv(output_dir / f"{column_name}.csv")
    print(f"Saved {column_name}.csv ({len(frame)} rows x {len(frame.columns)} columns)")